# XGBoost

In [ ]:
import pandas as pd
import os
import csv
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import numpy as np

In [ ]:
# Ouvre le CSV ../data/reddit_depression_dataset_cleaned_v2.csv
df = pd.read_csv("../data/reddit_depression_dataset_cleaned_v2.csv")

In [ ]:
df.head()

In [ ]:
# 1. Split (même que LogisticRegression)
df = df.dropna(subset=["label"]).copy()
text_series = df["clean_text"].fillna("").astype(str)
label_series = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    text_series, label_series, test_size=0.2, random_state=42, stratify=label_series
)

In [ ]:
# 2. Vectorisation TF-IDF (textes deja nettoyes en cellule 5)
vectorizer = TfidfVectorizer(
    max_features=10000, ngram_range=(1, 2), stop_words="english"
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
# 3. XGBoost avec paramètres optimisés
xgb_model = xgb.XGBClassifier(
    n_estimators=300,  # Plus d'arbres
    max_depth=8,  # Plus profond
    learning_rate=0.05,  # Plus lent mais précis
    subsample=0.8,  # Régularisation
    colsample_bytree=0.8,  # Diversité features
    scale_pos_weight=4.2,  # Ratio 80/20 inversé (0.8078/0.1922)
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)

In [ ]:
print("Entraînement XGBoost...")
xgb_model.fit(X_train_tfidf, y_train, verbose=False)

In [ ]:
# 4. Évaluation
y_pred_xgb = xgb_model.predict(X_test_tfidf)
print("=== XGBoost vs LogisticRegression ===")
print("XGBoost Classification Report :")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# 5. Sauvegarde
joblib.dump(xgb_model, "../models/xgb_depression_classifier.pkl")
joblib.dump(vectorizer, "../models/xgb_tfidf_vectorizer.pkl")
print("✅ XGBoost sauvegardé !")

In [ ]:
# 6. Top features (interprétabilité)
importances = xgb_model.feature_importances_
top_idx = np.argsort(importances)[-15:]
top_features = vectorizer.get_feature_names_out()[top_idx]
print("Top 15 mots/phrases prédictifs :")
for i, feat in enumerate(reversed(top_features)):
    print(f"{i + 1}. '{feat}' (importance: {importances[top_idx[-i - 1]]:.3f})")

| Modèle   | F1 Dépressifs | Recall Dépressifs | Precision | Accuracy |
| -------- | ------------- | ----------------- | --------- | -------- |
| Logistic | 83%           | 91%               | 76%       | 93%      |
| XGBoost  | 80%           | 87%               | 73%       | 91%      |

Verdict : LogisticRegression gagne légèrement sur ce dataset ! C’est fréquent avec TF-IDF + textes courts : la régression logistique est très efficace et plus simple

XGBoost garde ses avantages
Malgré le léger recul F1 :

Plus robuste aux outliers/noise

Feature importance naturelle (mots clés identifiés)

Meilleure généralisation sur textes variés